In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults  ## search tool  used for internet search
from langchain import hub
from langsmith import Client
from langchain.tools import tool


In [2]:
from langchain.agents import create_react_agent, AgentExecutor   ## react ----> Reasoning and action == ReAct this is langchain agent to create our agent


In [6]:
## load env variable

# os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")



In [7]:
search_tool= TavilySearchResults(max_results=2)

/var/folders/p4/697jx8ns5d3_5hf0yn67fywh0000gn/T/ipykernel_75143/4218638880.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_tool= TavilySearchResults(max_results=2)


In [8]:
@tool         ##if i want to use custom function as tool then i have to use decorator (@) now it become custom tool
def get_weather_data(city:str) -> str :
        """
        Fetch current weather information for a city.
        """

        url=(
                f"https://api.weatherstack.com/current?"
                f"access_key={WEATHERSTACK_API_KEY}&query={city}"
        )
        print(url)

        response = requests.get(url)

        data = response.json()

        if "current" not in data:
                return f"Could not fetch weather data for {city}"
        
        print(data['current'])

        return (
                f"City:{city}\n"
                f"Temperature: {data['current']['temperature']}°C\n"  ##option+shift+8 for degree sign
                f"Weather: {data['current']['weather_descriptions'][0]}\n"
                f"Humidity: {data['current']['humidity']}%"
        )

In [9]:
result = get_weather_data("delhi")
result

##"https://api.weatherstack.com/current?access_key=9b5d39aa8ad1676a188f827425ae0c71&query=delhi"

/var/folders/p4/697jx8ns5d3_5hf0yn67fywh0000gn/T/ipykernel_75143/340293920.py:1: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = get_weather_data("delhi")


https://api.weatherstack.com/current?access_key=9b5d39aa8ad1676a188f827425ae0c71&query=delhi
{'observation_time': '07:36 PM', 'temperature': 23, 'weather_code': 116, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0002_sunny_intervals.png'], 'weather_descriptions': ['Partly Cloudy '], 'astro': {'sunrise': '07:15 AM', 'sunset': '07:10 PM', 'moonrise': '07:22 PM', 'moonset': '08:18 AM', 'moon_phase': 'Waning Gibbous', 'moon_illumination': 100}, 'air_quality': {'co': '122', 'no2': '3', 'o3': '86', 'so2': '3', 'pm2_5': '4.4', 'pm10': '4.5', 'us-epa-index': '1', 'gb-defra-index': '1'}, 'wind_speed': 21, 'wind_degree': 36, 'wind_dir': 'NE', 'pressure': 1013, 'precip': 0, 'humidity': 32, 'cloudcover': 33, 'feelslike': 18, 'uv_index': 4, 'visibility': 10, 'is_day': 'yes'}


'City:delhi\nTemperature: 23°C\nWeather: Partly Cloudy \nHumidity: 32%'

In [10]:
result = search_tool.invoke("What is the current score of India in cricket today match")
result

[{'title': 'Live Cricket Score, Live Match Commentary, Live Score Today (28th Sep 2026), Match Scorecard T20, Test, ODI, IPL - India Today',
  'url': 'https://www.indiatoday.in/live-score',
  'content': "View All\n\n1\n\n \n\n#### Shubman Gill\n\nIndia\n\n801 Points\n\n| POS | PLAYER | TEAM | POINTS |\n ---  --- |\n| 2 | Daryl Mitchell | New Zealand | 794 |\n| 3 | Virat Kohli | India | 767 |\n| 4 | Rohit Sharma | India | 758 |\n| 5 | Ibrahim Zadran | Afghanistan | 719 |\n| 6 | Babar Azam | Pakistan | 689 |\n| 7 | Joe Root | England | 675 |\n\nView All\n\nAdvertisement\n\n## TOP CRICKET STORIES\n\n.\n\nIn which stadium is the match being played?\n\nCAY vs BER — Jimmy Powell Oval, George Town\n\nWhen did the match start, and when will it end?\n\nCAY vs BER — Start: 00:00(IST), Expected End: 9/28/2026\n\nWhat is the current match situation — first innings or second innings?\n\nCAY vs BER — Currently in First innings.\n\nWhat is live cricket score? [...] BAN\n\nVS\n\nMAS\n\nKorogi Sports P

In [12]:
## single agent system now

## now initailize llm

llm= ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0,
        api_key=OPENAI_API_KEY
)

In [13]:
llm_1=ChatGroq(
        model="openai/gpt-oss-120b",
        api_key=GROQ_API_KEY
)

In [14]:
## to check llm is working or not we can invoke it

response= llm_1.invoke("what is your last trained year")
response

AIMessage(content='My training data includes information up through June\u202f2024.', additional_kwargs={'reasoning_content': 'The user asks: "what is your last trained year". We need to answer. According to system message: "You are ChatGPT, a large language model trained by OpenAI. Knowledge cutoff: 2024-06". So answer: My training data goes up to June 2024. So respond accordingly. Also note no disallowed content. Provide concise answer.'}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 77, 'total_tokens': 174, 'completion_time': 0.200868718, 'completion_tokens_details': {'reasoning_tokens': 76}, 'prompt_time': 0.002980799, 'prompt_tokens_details': None, 'queue_time': 0.396053479, 'total_time': 0.203849517}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a0e45e-caf3-7f13-ae35-0b82076349b8-0', usage_metadata={'input_tokens': 77, 'output_tokens':

In [15]:
## to download prompt from langchain hub (need langsmith) or you can write it manually


client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [16]:
tools=[search_tool, get_weather_data]

In [18]:
## now we create the agent

agent = create_react_agent(
        llm=llm_1,
        tools=tools,
        prompt=prompt
)

In [19]:
## agent executor

agent_executor= AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True ,   ## to see the log of plans and all (to able to see execution process)
        handle_parsing_errors=True    
)

In [20]:
## ro run

response = agent_executor.invoke({
        "input": (
                "Find the capital of India"
                "and then find its current weather."
        )
})



> Entering new AgentExecutor chain...
Question: Find the capital of India and then find its current weather.  
Thought: I need to determine the capital city of India first. I will search for that information.  
Action: tavily_search_results_json  
Action Input: capital of India  [{'title': 'New Delhi', 'url': 'https://en.wikipedia.org/wiki/New_Delhi', 'content': 'Edit links\n\n Article\n Talk\n\n Read\n View source\n View history\n\nTools\n\nActions\n\n Read\n View source\n View history\n\nGeneral\n\n What links here\n Related changes\n Upload file\n Permanent link\n Page information\n Cite this page\n Get shortened URL\n Switch to legacy parser\n\nPrint/export\n\n Download as PDF\n Printable version\n\nIn other projects\n\n Abstract Wikipedia\n Wikimedia Commons\n Wikinews\n Wikiquote\n Wikivoyage\n Wikidata item\n\nAppearance\n\nCoordinates: 28°36′50″N 77°12′32″E\ufeff / \ufeff28.6139°N 77.2089°E\ufeff / 28.6139; 77.2089\n\nPage semi-protected\n\nFrom Wikipedia, the free encycloped

In [21]:
print(response["output"])

The capital of India is **New Delhi**. According to the latest weather data, New Delhi is experiencing **24 °C** with **patchy rain nearby** and a humidity of **74 %**.


Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: 


                llm
                 |

user --------> agent --------->output
|
tavily
